# Notebook 44 - Hardware cost proxy for the notebook 40 structures (40, 70, 80 and 90%)

**Why this notebook.** Notebook 36 measured size, memory footprint and single-thread CPU latency for the frozen 40% structures only. Notebook 40 then showed that the shallow and deep CICIoT2023 detectors recover to within 0.02 harm-weighted severity of dense at 90% realised MAC reduction, but the cost side of that result is so far reported by realised MACs alone. This notebook measures the committed 70, 80 and 90% structures from notebook 40 exactly the way notebook 36 measured the 40% ones, so cost and headroom are reported at the same budgets.

**What it does.** CPU runtime, about fifteen minutes, no data is opened. (1) Rebuilds every structure in `results/saber/40_compression_headroom/N40_*_removed_groups.csv` (two networks, five scoring rules, four budgets) and the two dense networks with random weights; latency and memory depend on the graph, not the weights. (2) Checks each rebuilt model against `N40_structures.csv` (parameter count exact, realised reduction within 1e-6) and the 40% rows against notebook 36's `hardware_proxy.csv` (parameters and MACs exact). (3) Measures, on one CPU thread, the parameter and activation footprint, the serialised size, and the median and 99th-percentile per-batch latency at batch 1 and batch 256 over 200 timed passes after 20 warm-ups; the whole sweep is repeated in three rounds and each model's latency is the median over rounds, with the between-round spread kept as a noise indicator. (4) Writes `H44_hardware_headroom.csv` with every measurement and its ratio to the dense network of the same architecture, a summary JSON, and a 600 dpi figure.

**Reporting, fixed in stage 2.** Descriptive, no pass or fail: per architecture and budget, the range across the five rules of each cost ratio, placed beside the realised MAC ratio. The notebook 36 reading (batch-256 latency tracks MACs, batch-1 latency is dominated by framework overhead) is re-examined at 70 to 90% and reported either way.

In [ ]:
# Stage 1 - bootstrap (CPU runtime is sufficient; a GPU is not used)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, time, gc, platform, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters
from src.saber.deep_model import DeepCNN1D

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError as e:                     # already started; intra-op single-threading is what matters here
    print("interop threads left at default:", e)
R = REPO / "results/saber"
N40 = R / "40_compression_headroom"
N36 = R / "36_hardware_proxy"
OUT = R / "44_hardware_headroom"; OUT.mkdir(parents=True, exist_ok=True)
assert (N40 / "N40_structures.csv").exists() and (N36 / "hardware_proxy.csv").exists(), "upstream result files missing; pull the repo"
CPU = platform.processor() or platform.machine()
try:
    CPU = [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
except Exception:
    pass
print("threads:", torch.get_num_threads(), "| cpu:", CPU, "| torch:", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
BUDGETS = [0.4, 0.7, 0.8, 0.9]
ARCHS = ["shallow", "deep"]
N_FEATURES, N_CLASSES = 41, 34
MIN_W = {"shallow": int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"]), "deep": 8}   # as NB40
WARMUP, TIMED, ROUNDS = 20, 200, 3
PREREG = {
    "notebook": "44_hardware_headroom",
    "question": ("what do the NB40 structures at 70, 80 and 90% realised MAC reduction cost on a single CPU thread, measured the way "
                 "NB36 measured the 40% structures: serialised size, parameter and activation footprint, and per-batch latency at "
                 "batch 1 and batch 256"),
    "motivation": ("NB36 timed only the frozen 40% structures. NB40 showed that the shallow and deep CICIoT2023 detectors recover to "
                   "within 0.02 harm-weighted severity of dense at 90% after two full epochs; the cost side of that result is so far "
                   "reported by realised MACs alone. This notebook measures the committed 70/80/90% structures so that cost and "
                   "headroom are reported at the same budgets"),
    "design": {
        "corpus": (f"CICIoT2023 only ({N_FEATURES} features, {N_CLASSES} classes); no data is opened: latency and memory depend on the "
                   "graph, not the weights, so every model is rebuilt from its committed structure with random weights "
                   "(torch.manual_seed(0)), exactly as NB36"),
        "models": ("per architecture the dense network plus 5 scoring rules x 4 budgets (40, 70, 80, 90%) = 21 models, 42 in all; "
                   "structures read from results/saber/40_compression_headroom/N40_{arch}_{method}_b{budget}_removed_groups.csv; "
                   f"per-layer floors shallow {MIN_W['shallow']} / deep {MIN_W['deep']} as in NB40; the shallow network is Conv 64, BN, "
                   "Conv 128, BN, GAP, linear and the deep network is src.saber.deep_model.DeepCNN1D, as in NB40"),
        "device": "single CPU thread of the Colab runtime (x86), reported as a PROXY for constrained hardware; no edge board was available",
        "measurements": {
            "latency": (f"median and 99th percentile of per-batch wall time over {TIMED} timed forward passes after {WARMUP} warm-up "
                        f"passes, for batch sizes 1 and 256, eval mode, no grad; the sweep over all 42 models is repeated in {ROUNDS} rounds "
                        "(each round: shallow dense, shallow students, deep dense, deep students) and the reported latency of a model is the "
                        "median over rounds of its per-round median (p99 likewise); the between-round spread (max minus min of the per-round "
                        "medians, divided by the reported median) is kept per model as a noise indicator and reported for the dense networks"),
            "memory": ("parameter and buffer bytes, plus the summed bytes of every module output in one batch-256 forward pass captured "
                       "by forward hooks (an upper bound on live activations)"),
            "size": "serialised state_dict size in KB",
            "flops": "realised multiply-accumulates per flow from the surgery profiler, the same cost axis as NB40 and NB36"},
    },
    "checks_that_stop_the_notebook": [
        "the parameter count of every rebuilt model equals the parameters column of N40_structures.csv exactly",
        "the realised reduction (1 - profiler MACs / dense MACs) agrees with N40_structures.csv realised_reduction within 1e-6",
        "the 40% rows' parameters and MACs equal NB36 hardware_proxy.csv exactly (same frozen structures)",
    ],
    "reporting": ("descriptive, no pass/fail: per architecture and budget, the range across the five rules of the size, parameter, "
                  "activation, batch-1 and batch-256 latency ratios (student / dense), placed beside the realised MAC ratio; the 40% "
                  "latency ratios are reported beside NB36's but are not required to match (different runtime day); the NB36 reading "
                  "that batch-256 latency tracks MACs while batch-1 latency is dominated by framework overhead is re-examined at "
                  "70 to 90% and reported either way"),
    "no_test_access": True,
}
pre_path = OUT / "H44_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2)); print("per-layer floors:", MIN_W)

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/44_hardware_headroom/H44_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "H44_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB44 pre-registration, committed before any result: hardware cost proxy for the NB40 structures at 40/70/80/90% "
                        "(CICIoT2023, shallow and deep, five scoring rules, 42 models rebuilt from the committed removed-groups files with "
                        "random weights). Same measurements as NB36 on one CPU thread: parameter/activation footprint, serialised size, "
                        "median and p99 latency at batch 1 and 256 over 200 timed passes after 20 warm-ups in 3 rounds (median over rounds, between-round spread kept), realised MACs from the surgery "
                        "profiler. Stops unless parameters match N40_structures.csv exactly and realised reduction within 1e-6; 40% rows "
                        "must equal NB36 in parameters and MACs. Descriptive reporting, no pass/fail. No data opened, no test access."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - model factory, structure loader and self-test against N40_structures.csv
class CNN1D(nn.Module):
    '''The shallow CICIoT2023 detector of NB40 (Conv 64, BN, Conv 128, BN, GAP, linear); module paths conv.0 and conv.3.'''
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


def structure_path(arch, method, budget):
    return N40 / f"N40_{arch}_{method}_b{int(round(100 * budget))}_removed_groups.csv"


def build(arch, method=None, budget=None):
    '''Dense network, or the student rebuilt from its committed structure, with random weights (seed 0).'''
    torch.manual_seed(0)
    dense = (CNN1D(N_CLASSES) if arch == "shallow" else DeepCNN1D(N_CLASSES)).eval()
    example = torch.randn(8, N_FEATURES)
    if method is None:
        return dense, example
    rm = pd.read_csv(structure_path(arch, method, budget))
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(dense, pm, example, minimum_remaining_per_layer=MIN_W[arch])
    return st.eval(), example


def widths(model):
    return "/".join(str(m.out_channels) for m in model.modules() if isinstance(m, nn.Conv1d))


for arch in ARCHS:
    for m in METHODS:
        for b in BUDGETS:
            assert structure_path(arch, m, b).exists(), f"missing structure {arch}/{m}/{b}"
print("all 40 committed structures present")

STRUCT = pd.read_csv(N40 / "N40_structures.csv").set_index(["architecture", "method", "budget"])
NB36 = pd.read_csv(N36 / "hardware_proxy.csv"); NB36 = NB36[NB36.corpus == "ciciot2023"].set_index(["architecture", "model"])
M0, checks = {}, []
for arch in ARCHS:
    dense, ex = build(arch)
    M0[arch] = profile_forward_flops(dense, ex)["flops_per_item"]
    n_dense = count_parameters(dense)
    assert n_dense == int(NB36.loc[(arch, "dense"), "parameters"]) and abs(M0[arch] - float(NB36.loc[(arch, "dense"), "flops_per_flow"])) < 1e-6, \
        f"dense {arch} differs from NB36: {n_dense} params, {M0[arch]:.0f} MACs"
    print(f"dense {arch}: widths {widths(dense)}, {n_dense:,} parameters, {M0[arch]:,.0f} MACs per flow (equal to NB36)")
    for m in METHODS:
        for b in BUDGETS:
            st, ex = build(arch, m, b)
            n = count_parameters(st); rr = 1.0 - profile_forward_flops(st, ex)["flops_per_item"] / M0[arch]
            ref = STRUCT.loc[(arch, m, b)]
            ok_n, ok_r, ok_w = n == int(ref.parameters), abs(rr - float(ref.realised_reduction)) < 1e-6, widths(st) == str(ref.widths)
            checks.append({"architecture": arch, "method": m, "budget": b, "widths": widths(st), "parameters": n, "realised_reduction": rr,
                           "n40_parameters": int(ref.parameters), "n40_realised_reduction": float(ref.realised_reduction),
                           "parameters_match": ok_n, "reduction_match": ok_r, "widths_match": ok_w})
            del st
checks = pd.DataFrame(checks)
print(checks[["architecture", "method", "budget", "widths", "parameters", "n40_parameters", "realised_reduction", "n40_realised_reduction"]].round(6).to_string(index=False))
assert checks.parameters_match.all() and checks.reduction_match.all() and checks.widths_match.all(), \
    "a rebuilt structure differs from N40_structures.csv; stop here and report it"
print("\nself-test passed: all 40 rebuilt structures equal N40_structures.csv in widths, parameters and realised reduction")

In [ ]:
# Stage 4 - measurement on one CPU thread (same procedure as NB36)
def footprint_mb(model, n_features, bs=256):
    '''Parameter+buffer bytes plus the summed bytes of every module output in one forward pass (an upper bound on live activations).'''
    acts = []
    def hook(_m, _i, o):
        if torch.is_tensor(o):
            acts.append(o.numel() * o.element_size())
    hs = [m.register_forward_hook(hook) for m in model.modules() if len(list(m.children())) == 0]
    with torch.no_grad():
        model(torch.randn(bs, n_features))
    for h in hs:
        h.remove()
    params = sum(p.numel() * p.element_size() for p in model.parameters()) + sum(b.numel() * b.element_size() for b in model.buffers())
    return params / 1e6, sum(acts) / 1e6


def measure(model, n_features):
    out = {}
    out["params_mb"], out["activations_mb_b256"] = footprint_mb(model, n_features)
    with torch.no_grad():
        for bs in (1, 256):
            x = torch.randn(bs, n_features)
            for _ in range(WARMUP):
                model(x)
            t = []
            for _ in range(TIMED):
                t0 = time.perf_counter(); model(x); t.append(time.perf_counter() - t0)
            t = np.array(t) * 1e3
            out[f"latency_ms_b{bs}_median"] = float(np.median(t)); out[f"latency_ms_b{bs}_p99"] = float(np.percentile(t, 99))
    sd = model.state_dict(); tmp = OUT / "_tmp.pt"; torch.save(sd, tmp); out["size_kb"] = tmp.stat().st_size / 1024.0; tmp.unlink()
    return out


PLAN = [(arch, method, budget) for arch in ARCHS for method, budget in [(None, None)] + [(m, b) for b in BUDGETS for m in METHODS]]
static, per_round = {}, []
t_start = time.time()
for rnd in range(ROUNDS):
    for arch, method, budget in PLAN:
        model, example = build(arch, method, budget)
        key = (arch, "dense" if method is None else method, 0.0 if method is None else budget)
        if rnd == 0:
            flops = profile_forward_flops(model, example)["flops_per_item"]
            pm, am = footprint_mb(model, N_FEATURES)
            sd = model.state_dict(); tmp = OUT / "_tmp.pt"; torch.save(sd, tmp); size_kb = tmp.stat().st_size / 1024.0; tmp.unlink()
            static[key] = {"widths": widths(model), "parameters": int(count_parameters(model)), "flops_per_flow": float(flops),
                           "realised_reduction": float(1.0 - flops / M0[arch]), "params_mb": pm, "activations_mb_b256": am, "size_kb": size_kb}
        meas = measure(model, N_FEATURES)
        per_round.append({"round": rnd, "architecture": key[0], "model": key[1], "budget": key[2], **meas})
        print(f"round {rnd} {arch:7s} {key[1]:9s} b{int(round(100 * key[2])):2d}: b1 {meas['latency_ms_b1_median']:.3f} ms (p99 {meas['latency_ms_b1_p99']:.3f}) | "
              f"b256 {meas['latency_ms_b256_median']:.2f} ms (p99 {meas['latency_ms_b256_p99']:.2f})")
        del model; gc.collect()
    print(f"-- round {rnd} done, {time.time() - t_start:.0f} s elapsed")
per_round = pd.DataFrame(per_round); per_round.to_csv(OUT / "H44_rounds.csv", index=False)

LAT = ["latency_ms_b1_median", "latency_ms_b1_p99", "latency_ms_b256_median", "latency_ms_b256_p99"]
rows = []
for key, st in static.items():
    g = per_round[(per_round.architecture == key[0]) & (per_round.model == key[1]) & (per_round.budget == key[2])]
    row = {"corpus": "ciciot2023", "architecture": key[0], "model": key[1], "budget": key[2], **st}
    for c in LAT:
        row[c] = float(g[c].median())
    for bs in (1, 256):
        m = g[f"latency_ms_b{bs}_median"]; row[f"latency_ms_b{bs}_round_spread"] = float((m.max() - m.min()) / m.median())
    rows.append(row)
res = pd.DataFrame(rows)
for key in ["flops_per_flow", "latency_ms_b1_median", "latency_ms_b1_p99", "latency_ms_b256_median", "latency_ms_b256_p99", "size_kb", "parameters", "activations_mb_b256"]:
    dense = res[res.model == "dense"].set_index("architecture")[key]
    res[f"{key}_ratio"] = res.apply(lambda r: r[key] / dense.loc[r.architecture], axis=1)
res.to_csv(OUT / "H44_hardware_headroom.csv", index=False)
print(f"\nmeasured {len(res)} models x {ROUNDS} rounds in {time.time() - t_start:.0f} s; latency = median over rounds")
print(res[["architecture", "model", "budget", "parameters", "flops_per_flow", "size_kb", "latency_ms_b1_median", "latency_ms_b256_median",
           "latency_ms_b1_round_spread", "latency_ms_b256_round_spread"]].round(3).to_string(index=False))
print("\nratios (student / dense):")
print(res[res.model != "dense"][["architecture", "model", "budget", "flops_per_flow_ratio", "parameters_ratio", "size_kb_ratio",
                                 "activations_mb_b256_ratio", "latency_ms_b1_median_ratio", "latency_ms_b256_median_ratio"]].round(3).to_string(index=False))

In [ ]:
# Stage 5 - cross-check against NB36, summary and figure
res = pd.read_csv(OUT / "H44_hardware_headroom.csv")
students = res[res.model != "dense"].copy()

# (a) the 40% rows are the same frozen structures NB36 measured: parameters and MACs must be equal; latency ratios are reported beside NB36's
x = students[students.budget == 0.4].set_index(["architecture", "model"])
cross = pd.DataFrame({"parameters_nb44": x.parameters, "parameters_nb36": NB36.parameters.reindex(x.index),
                      "flops_nb44": x.flops_per_flow, "flops_nb36": NB36.flops_per_flow.reindex(x.index),
                      "b1_ratio_nb44": x.latency_ms_b1_median_ratio, "b1_ratio_nb36": NB36.latency_ms_b1_median_ratio.reindex(x.index),
                      "b256_ratio_nb44": x.latency_ms_b256_median_ratio, "b256_ratio_nb36": NB36.latency_ms_b256_median_ratio.reindex(x.index),
                      "size_ratio_nb44": x.size_kb_ratio, "size_ratio_nb36": NB36.size_kb_ratio.reindex(x.index)}).reset_index()
assert (cross.parameters_nb44 == cross.parameters_nb36).all() and np.allclose(cross.flops_nb44, cross.flops_nb36), "40% rows differ from NB36 in parameters or MACs"
cross.to_csv(OUT / "H44_nb36_crosscheck.csv", index=False)
print("40% rows equal NB36 in parameters and MACs; latency ratios side by side (this run / NB36):")
print(cross[["architecture", "model", "b1_ratio_nb44", "b1_ratio_nb36", "b256_ratio_nb44", "b256_ratio_nb36", "size_ratio_nb44", "size_ratio_nb36"]].round(3).to_string(index=False))
b256_abs_diff = float((cross.b256_ratio_nb44 - cross.b256_ratio_nb36).abs().max())
b1_abs_diff = float((cross.b1_ratio_nb44 - cross.b1_ratio_nb36).abs().max())

# (b) per architecture and budget: range across the five rules of every ratio
KEYS = {"flops": "flops_per_flow_ratio", "parameters": "parameters_ratio", "size": "size_kb_ratio", "activations_b256": "activations_mb_b256_ratio",
        "latency_b1": "latency_ms_b1_median_ratio", "latency_b256": "latency_ms_b256_median_ratio"}
per_budget = {}
for (arch, b), g in students.groupby(["architecture", "budget"]):
    per_budget[f"{arch}/b{int(round(100 * b))}"] = {k: [round(float(g[c].min()), 4), round(float(g[c].max()), 4)] for k, c in KEYS.items()}
    per_budget[f"{arch}/b{int(round(100 * b))}"]["by_rule"] = {r.model: {k: round(float(getattr(r, c)), 4) for k, c in KEYS.items()} for r in g.itertuples()}
# does batch-256 latency track MACs more closely than batch-1 at 70-90%? (reported, not scored)
hi = students[students.budget >= 0.7]
track = {"b256_minus_flops_ratio_range": [round(float((hi.latency_ms_b256_median_ratio - hi.flops_per_flow_ratio).min()), 4),
                                         round(float((hi.latency_ms_b256_median_ratio - hi.flops_per_flow_ratio).max()), 4)],
         "b1_minus_flops_ratio_range": [round(float((hi.latency_ms_b1_median_ratio - hi.flops_per_flow_ratio).min()), 4),
                                       round(float((hi.latency_ms_b1_median_ratio - hi.flops_per_flow_ratio).max()), 4)],
         "b256_closer_than_b1_share": round(float(((hi.latency_ms_b256_median_ratio - hi.flops_per_flow_ratio).abs()
                                                  < (hi.latency_ms_b1_median_ratio - hi.flops_per_flow_ratio).abs()).mean()), 3)}
summary = {"notebook": "44_hardware_headroom", "cpu": CPU, "torch": torch.__version__, "threads": int(torch.get_num_threads()),
           "models_measured": int(len(res)), "per_budget_ratio_ranges_over_rules": per_budget, "tracking_at_70_90": track,
           "dense_absolute": {r.architecture: {"parameters": int(r.parameters), "size_kb": round(float(r.size_kb), 1),
                                               "latency_ms_b1_median": round(float(r.latency_ms_b1_median), 3),
                                               "latency_ms_b256_median": round(float(r.latency_ms_b256_median), 2),
                                               "activations_mb_b256": round(float(r.activations_mb_b256), 2)} for r in res[res.model == "dense"].itertuples()},
           "nb36_crosscheck": {"parameters_and_macs_equal": True, "max_abs_diff_b1_ratio": round(b1_abs_diff, 4), "max_abs_diff_b256_ratio": round(b256_abs_diff, 4)},
           "round_spread": {"rounds": int(ROUNDS),
                            "dense": {r.architecture: {"b1": round(float(r.latency_ms_b1_round_spread), 4), "b256": round(float(r.latency_ms_b256_round_spread), 4)}
                                      for r in res[res.model == "dense"].itertuples()},
                            "students_max": {"b1": round(float(students.latency_ms_b1_round_spread.max()), 4), "b256": round(float(students.latency_ms_b256_round_spread.max()), 4)},
                            "students_median": {"b1": round(float(students.latency_ms_b1_round_spread.median()), 4), "b256": round(float(students.latency_ms_b256_round_spread.median()), 4)}},
           "prereg": json.load(open(OUT / "H44_PREREGISTRATION.json"))}
(OUT / "H44_summary.json").write_text(json.dumps(summary, indent=2))
print("\nper budget, range over the five rules (student / dense):")
for k, v in per_budget.items():
    print(f"  {k:12s} MACs {v['flops'][0]:.3f}-{v['flops'][1]:.3f} | size {v['size'][0]:.3f}-{v['size'][1]:.3f} | act {v['activations_b256'][0]:.3f}-{v['activations_b256'][1]:.3f} | "
          f"b1 {v['latency_b1'][0]:.3f}-{v['latency_b1'][1]:.3f} | b256 {v['latency_b256'][0]:.3f}-{v['latency_b256'][1]:.3f}")
print("tracking at 70-90%:", json.dumps(track)); print("round spread:", json.dumps(summary["round_spread"]))

# figure: cost ratios against the realised MAC ratio, 600 dpi
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
COL = {"random": "#999999", "magnitude": "#E69F00", "taylor": "#56B4E9", "fisher": "#009E73", "saber_v2": "#CC79A7"}
NAME = {"random": "random", "magnitude": "magnitude", "taylor": "Taylor", "fisher": "Fisher", "saber_v2": "leverage"}
MARK = {"shallow": "o", "deep": "^"}
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.0))
for ax, key, lab in [(axes[0], "size_kb_ratio", "size ratio"), (axes[1], "latency_ms_b256_median_ratio", "latency ratio, batch 256"),
                     (axes[2], "latency_ms_b1_median_ratio", "latency ratio, batch 1")]:
    ax.plot([0, 1.05], [0, 1.05], color="#B8B8B8", lw=0.6, ls=":", zorder=1)
    for arch in ARCHS:
        for m in METHODS:
            g = students[(students.architecture == arch) & (students.model == m)].sort_values("budget")
            ax.plot(g.flops_per_flow_ratio, g[key], color=COL[m], lw=0.6, alpha=0.6, zorder=2)
            ax.scatter(g.flops_per_flow_ratio, g[key], color=COL[m], marker=MARK[arch], s=12, lw=0.4, edgecolor="white", zorder=3,
                       label=f"{NAME[m]}, {arch}" if key == "size_kb_ratio" else None)
    ax.set_xlim(0, 1.05); ax.set_ylim(0, max(1.1, 1.05 * float(students[key].max()))); ax.set_xlabel("realised MAC ratio (student / dense)"); ax.set_ylabel(lab)
    ax.set_xticks([0.1, 0.2, 0.3, 0.6, 1.0]); ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 1.08), handletextpad=0.3, columnspacing=0.9)
fig.tight_layout()
fig.savefig(OUT / "H44_cost_vs_reduction.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "H44_cost_vs_reduction.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 6 - commit the results (standard commit cell; the message is filled in from the summary)
s = json.load(open(OUT / "H44_summary.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/44_hardware_headroom/H44_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
pb, tr, d = s["per_budget_ratio_ranges_over_rules"], s["tracking_at_70_90"], s["dense_absolute"]
def rng(arch, b, k, d=2):
    v = pb[f"{arch}/b{b}"][k]; return f"{v[0]:.{d}f}-{v[1]:.{d}f}"
msg = (f"NB44 results (pre-registration {pre}): hardware cost proxy for the NB40 structures on one CPU thread ({s['cpu']}, torch {s['torch']}), "
       f"{s['models_measured']} models rebuilt from the committed structures with random weights; all 40 match N40_structures.csv exactly and the "
       f"40% rows equal NB36 in parameters and MACs (b256 latency ratio differs from NB36 by at most {s['nb36_crosscheck']['max_abs_diff_b256_ratio']:.3f}). "
       f"Student/dense ratios at 90%: size shallow {rng('shallow', 90, 'size')} / deep {rng('deep', 90, 'size')}; batch-256 latency shallow "
       f"{rng('shallow', 90, 'latency_b256')} / deep {rng('deep', 90, 'latency_b256')}; batch-1 latency shallow {rng('shallow', 90, 'latency_b1')} / deep "
       f"{rng('deep', 90, 'latency_b1')} against realised MAC ratio {rng('shallow', 90, 'flops', 3)} / {rng('deep', 90, 'flops', 3)}. At 70%: b256 "
       f"{rng('shallow', 70, 'latency_b256')} / {rng('deep', 70, 'latency_b256')}, b1 {rng('shallow', 70, 'latency_b1')} / {rng('deep', 70, 'latency_b1')}. "
       f"Over 70-90%, b256 ratio minus MAC ratio {tr['b256_minus_flops_ratio_range'][0]:+.2f}..{tr['b256_minus_flops_ratio_range'][1]:+.2f}, b1 ratio minus MAC ratio "
       f"{tr['b1_minus_flops_ratio_range'][0]:+.2f}..{tr['b1_minus_flops_ratio_range'][1]:+.2f}; b256 closer to MACs than b1 in {100 * tr['b256_closer_than_b1_share']:.0f}% of models. "
       f"Dense: shallow {d['shallow']['latency_ms_b1_median']:.3f} ms b1 / {d['shallow']['size_kb']:.0f} KB, deep {d['deep']['latency_ms_b1_median']:.3f} ms b1 / {d['deep']['size_kb']:.0f} KB. "
       f"Latency = median over {s['round_spread']['rounds']} rounds; between-round spread of the b256 median: dense shallow {s['round_spread']['dense']['shallow']['b256']:.3f}, "
       f"dense deep {s['round_spread']['dense']['deep']['b256']:.3f}, students median {s['round_spread']['students_median']['b256']:.3f} (max {s['round_spread']['students_max']['b256']:.3f}); "
       f"b1 students median {s['round_spread']['students_median']['b1']:.3f} (max {s['round_spread']['students_max']['b1']:.3f}). Descriptive, no pass/fail.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)